# UR_RTDE_Tests — UR3e (PolyScope X) + Hand-E bring-up checks

Sanity checks for the lab **UR3e on PolyScope X 10.12** (`192.168.1.4`), reusing
`UR3RealRobotPick` from `ur3_realrobot_dependencies.py`.

1. Connection test (arm via External Control URCapX)
2. Receive test
3. Hand-E control test (XML-RPC :49999)
4. Arm + hand to the `tucked` pose

**Arm control on PolyScope X:** the headless script `ur_rtde` normally uploads does **not**
run on PolyScope X. Instead we use the **External Control URCapX** (`use_ext_urcap=True`):
on the pendant, an *External Control* program — Host IP **192.168.1.89** (this PC), port
**50002** — must be **PLAYING**, and the robot must be in **Remote Control**.

**Gripper:** Robotiq URCapX XML-RPC server on `http://192.168.1.4:49999/`, slaveId 9.

In [ ]:
# Setup: backend env, imports, robot handle
import os
import platform
import sys

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=1"
if platform.system() == "Darwin":
    os.environ["MUJOCO_GL"] = "glfw"
    os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
else:
    os.environ.setdefault("MUJOCO_GL", "egl")

import time
import numpy as np

import rtde_control
import rtde_receive

# UR3RealRobotPick lives in the sibling robots/UR3e/ folder.
sys.path.insert(0, os.path.abspath("../UR3e"))
from ur3_realrobot_dependencies import UR3RealRobotPick

ROBOT_IP = "192.168.1.4"      # UR3e, PolyScope X 10.12
PC_HOST_IP = "192.168.1.89"   # this PC; set the External Control node's Host IP to this
# 'tucked' keyframe arm pose (6 arm joints) from
# my_ur3/xmls/mjx_single_cube_position_ur3.xml
Q_TUCKED = [0.0, -2.36, 2.36, -1.57, -1.57, 0.0]

# Sections 1-2 + 4 below talk to the raw rtde_receive / rtde_control interfaces
# directly (recv / ctrl) to test bare-metal control first. The wrapper handle is
# still created for the Hand-E gripper cells (section 3).
# use_ext_urcap=True -> arm driven via the External Control URCapX (PolyScope X)
robot = UR3RealRobotPick(host=ROBOT_IP, use_ext_urcap=True, ur_cap_port=50002)
print("ROBOT_IP", ROBOT_IP, "| raw rtde + wrapper handle (gripper) ready")

<cell_type>markdown</cell_type>## 1. Connection test — receive interface + current state

Opens the raw `rtde_receive` interface (joint feedback only — no control yet) and reads the
current state. This does **not** need the pendant program running; it just confirms the robot
is reachable on the network and streaming RTDE data.

In [ ]:
recv = rtde_receive.RTDEReceiveInterface(ROBOT_IP)
print("receive connected:", recv.isConnected())

q   = recv.getActualQ()           # 6 joint angles (rad)
qd  = recv.getActualQd()          # 6 joint velocities (rad/s)
tcp = recv.getActualTCPPose()     # [x, y, z, rx, ry, rz]
print("runtime state (2=PLAYING):", recv.getRuntimeState())
print("robot mode  (7=RUNNING):  ", recv.getRobotMode())
print("q   =", [round(v, 4) for v in q])
print("qd  =", [round(v, 4) for v in qd])
print("tcp =", [round(v, 4) for v in tcp])

<cell_type>markdown</cell_type>## 2. Receive test — live stream

Reads the raw `recv` interface a few times so we can eyeball that joint/TCP/force feedback is
sane and updating.

In [ ]:
for _ in range(5):
    print(
        "q =", [round(v, 4) for v in recv.getActualQ()],
        "| tcp =", [round(v, 4) for v in recv.getActualTCPPose()[:3]],
        "| force =", [round(v, 2) for v in recv.getActualTCPForce()],
    )
    time.sleep(0.2)

## 3. Hand-E control test (receive state, then open / close)

PolyScope X exposes the Robotiq gripper via an **XML-RPC server** on `http://<host>:49999/`
(NOT the legacy 63352 socket); this Hand-E is **slaveId 9**. The gripper rides this channel
separately from `servoJ`/`moveJ` (arm joints only).

**3a — receive only:** connect + activate, read raw native values (position **percent
0–100**, object flag, fault, activated). No motion. Reveals the native percent convention.

**3b — open / close:** drive with the direction-independent `open_gripper()`/`close_gripper()`
and record the native `pos_pct` at each extreme → locks the sim↔percent mapping.

Sim convention: **`0` = closed, `0.025` = open**. ⚠️ Don't command the XML-RPC server
above ~10 Hz.

In [ ]:
# 3a — connect + receive native gripper state (no motion yet)
robot.connect_gripper()  # XML-RPC :49999, slaveId 9, activateIfRequired

state = robot.read_gripper_state()
print("raw native values (URCapX, position in percent 0-100):")
for k in ["pos_pct", "obj_flag", "fault", "activated", "connected"]:
    print(f"  {k:10s} = {state[k]}")
print("derived (sim convention 0=closed, 0.025=open):")
print(f"  open_frac  = {state['open_frac']:.3f}  (1=open, 0=closed)")
print(f"  sim_finger = {state['sim_finger']:.4f}  (per-finger, target range 0..0.025)")
print("note: open_frac/sim_finger use the assumed pct mapping; confirm with 3b.")

In [ ]:
# 3b — open / close test, recording the native pos_pct at each extreme
print("opening...")
robot.open_gripper()       # direction-independent openGripper(9)
time.sleep(2.0)
open_pct = robot.read_gripper_state()["pos_pct"]
print("  pos_pct when OPEN :", open_pct)

print("closing...")
robot.close_gripper()      # direction-independent closeGripper(9)
time.sleep(2.0)
closed_pct = robot.read_gripper_state()["pos_pct"]
print("  pos_pct when CLOSED:", closed_pct)

print(f"\n=> set in ur3_realrobot_dependencies.__init__ if these differ from "
      f"the defaults (open=0, closed=100):")
print(f"     self._gripper_open_pct   = {open_pct}")
print(f"     self._gripper_closed_pct = {closed_pct}")
# Apply for the rest of this session so send_gripper(norm) maps correctly now:
robot._gripper_open_pct = float(open_pct)
robot._gripper_closed_pct = float(closed_pct)

<cell_type>markdown</cell_type>## 4. Control interface (External Control URCapX) + moveJ to `tucked`

Opens the raw `rtde_control` interface with `FLAG_USE_EXT_UR_CAP`, then sends a `moveJ` to the
`tucked` keyframe. The constructor **blocks until the pendant's External Control program
connects** — so **press Play on the pendant now** (External Control node → Host IP
`192.168.1.89`, port `50002`; robot in **Remote Control**).

`moveJ(q, speed, acceleration, asynchronous)`. The gripper close uses the wrapper handle
(separate XML-RPC channel — run section 3 first to connect it, or skip the close line).

In [ ]:
# BLOCKS until the pendant External Control program connects -> press Play first.
ctrl = rtde_control.RTDEControlInterface(
    ROBOT_IP,
    flags=rtde_control.RTDEControlInterface.FLAG_USE_EXT_UR_CAP,
)
print("control connected:", ctrl.isConnected())
print("program running:  ", ctrl.isProgramRunning())

# moveJ(q, speed, acceleration, asynchronous); blocking until converged.
ctrl.moveJ(Q_TUCKED, 0.4, 0.4, False)
print("reached tucked; q =", [round(v, 4) for v in recv.getActualQ()])

# Optional: close the Hand-E (tucked keyframe gripper ctrl = 0 -> closed).
# Requires section 3 to have connected the gripper first.
# robot.close_gripper()

## Teardown

Stops servo motion and closes the RTDE + gripper connections.

In [ ]:
# Raw control + receive interfaces
try:
    ctrl.servoStop()
    ctrl.stopScript()
    ctrl.disconnect()
except NameError:
    pass
try:
    recv.disconnect()
except NameError:
    pass

# Wrapper handle (gripper)
robot.disconnect()
print("disconnected")